# EDA con DuckDB

In [1]:
from pathlib import Path
import os
import duckdb
from IPython.display import display, Markdown
BASES = [Path.cwd(), *Path.cwd().parents]
CANDIDATOS = (p for base in BASES for p in (base, base/'persona-b', base/'Persona_B_Exploracion_EDA'))
ROOT = next((p for p in CANDIDATOS if (p/'sql/eda/00_vista.sql').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f'No encuentro sql/eda/00_vista.sql desde {Path.cwd()}. Extrae el contenido del ZIP en la raíz del repositorio o indica ROOT = Path(\"/ruta/a/tu/repositorio\").')
if not list((ROOT/'data/raw').glob('*/*/*.parquet')):
    raise FileNotFoundError(f'En {ROOT}/data/raw no hay Parquet bajo */*/*.parquet. Coloca sql/, notebooks/ y docs/ en la raíz del fork y ejecuta python scripts/download_data.py allí.')

os.chdir(ROOT)
print('Raíz detectada:', ROOT)
con = duckdb.connect(':memory:')
print('DuckDB', duckdb.__version__)
def tabla(columnas,filas):
    esc = lambda x: str('NULL' if x is None else x).replace('|','\\|').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(map(esc,columnas))+' |','| '+' | '.join(['---']*len(columnas))+' |']+['| '+' | '.join(map(esc,f))+' |' for f in filas])
def ejecutar(ruta):
    sql = (ROOT/ruta).read_text()
    salidas=[]
    for sentencia in (x.strip() for x in sql.split(';') if x.strip()):
        resultado=con.execute(sentencia)
        if resultado.description:
            columnas=[x[0] for x in resultado.description]
            filas=resultado.fetchall()
            salidas.append((columnas,filas))
            display(Markdown('### '+str(ruta)+'\n'+tabla(columnas,filas[:25])))
    return salidas

Raíz detectada: /Users/Camila/Desktop/CAMILA UNIVERSIDAD/8SEMESTRE/DataScience/Lab8-DS
DuckDB 1.5.6


In [2]:
con.execute((ROOT/'sql/eda/00_vista.sql').read_text())
resultados={}
for f in sorted((ROOT/'sql/eda').glob('0[1-6]_*.sql')):
    resultados[f.name]=ejecutar(f.relative_to(ROOT))

### sql/eda/01_temporal.sql
| tipo | anio_archivo | mes | viajes | dias_con_viajes |
| --- | --- | --- | --- | --- |
| green | 2024 | 2024-01-01 | 56555 | 31 |
| yellow | 2024 | 2024-01-01 | 2964617 | 31 |
| green | 2024 | 2024-02-01 | 53578 | 29 |
| yellow | 2024 | 2024-02-01 | 3007533 | 29 |
| green | 2024 | 2024-03-01 | 57451 | 31 |
| yellow | 2024 | 2024-03-01 | 3582611 | 31 |
| green | 2024 | 2024-04-01 | 56473 | 30 |
| yellow | 2024 | 2024-04-01 | 3514295 | 30 |
| green | 2024 | 2024-05-01 | 61007 | 31 |
| yellow | 2024 | 2024-05-01 | 3723843 | 31 |
| green | 2024 | 2024-06-01 | 54738 | 30 |
| yellow | 2024 | 2024-06-01 | 3539170 | 30 |
| green | 2024 | 2024-07-01 | 51820 | 31 |
| yellow | 2024 | 2024-07-01 | 3076876 | 31 |
| green | 2024 | 2024-08-01 | 51806 | 31 |
| yellow | 2024 | 2024-08-01 | 2979192 | 31 |
| green | 2024 | 2024-09-01 | 54422 | 30 |
| yellow | 2024 | 2024-09-01 | 3633025 | 30 |
| green | 2024 | 2024-10-01 | 56153 | 31 |
| yellow | 2024 | 2024-10-01 | 3833780 | 31 |
| green | 2024 | 2024-11-01 | 52214 | 30 |
| yellow | 2024 | 2024-11-01 | 3646372 | 30 |
| green | 2024 | 2024-12-01 | 53981 | 31 |
| yellow | 2024 | 2024-12-01 | 3668350 | 31 |
| green | 2025 | 2025-01-01 | 48288 | 31 |

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### sql/eda/02_caracteristicas.sql
| tipo | anio_archivo | viajes | distancia_media_filtrada | distancia_mediana_filtrada | duracion_mediana_min | pasajeros_media_filtrada |
| --- | --- | --- | --- | --- | --- | --- |
| green | 2024 | 660198 | 2.94 | 1.97 | 12.0 | 1.33 |
| yellow | 2024 | 41169664 | 3.43 | 1.8 | 13.0 | 1.35 |
| green | 2025 | 591354 | 3.19 | 2.06 | 13.0 | 1.31 |
| yellow | 2025 | 48722573 | 3.5 | 1.9 | 13.0 | 1.3 |
| green | 2026 | 337100 | 3.36 | 2.14 | 13.0 | 1.32 |
| yellow | 2026 | 29703338 | 3.51 | 1.92 | 14.0 | 1.25 |

### sql/eda/03_pago.sql
| tipo | anio_archivo | tipo_pago | viajes | porcentaje | propina_media_no_negativa |
| --- | --- | --- | --- | --- | --- |
| green | 2024 | 1 | 454677 | 68.87 | 3.55 |
| green | 2024 | 2 | 175005 | 26.51 | 0.0 |
| green | 2024 | 3 | 4598 | 0.7 | 0.03 |
| green | 2024 | 4 | 1561 | 0.24 | 0.02 |
| green | 2024 | 5 | 29 | 0.0 | 0.0 |
| green | 2024 | NULL | 24328 | 3.68 | 3.58 |
| yellow | 2024 | 0 | 4091232 | 9.94 | 0.76 |
| yellow | 2024 | 1 | 30452126 | 73.97 | 4.37 |
| yellow | 2024 | 2 | 5540067 | 13.46 | 0.0 |
| yellow | 2024 | 3 | 291741 | 0.71 | 0.03 |
| yellow | 2024 | 4 | 794494 | 1.93 | 0.06 |
| yellow | 2024 | 5 | 4 | 0.0 | 0.0 |
| green | 2025 | 1 | 406600 | 68.76 | 3.69 |
| green | 2025 | 2 | 129823 | 21.95 | 0.0 |
| green | 2025 | 3 | 3784 | 0.64 | 0.05 |
| green | 2025 | 4 | 1244 | 0.21 | 0.01 |
| green | 2025 | 5 | 23 | 0.0 | 0.0 |
| green | 2025 | NULL | 49880 | 8.43 | 1.51 |
| yellow | 2025 | 0 | 11611894 | 23.83 | 0.33 |
| yellow | 2025 | 1 | 31053982 | 63.74 | 4.34 |
| yellow | 2025 | 2 | 4654334 | 9.55 | 0.0 |
| yellow | 2025 | 3 | 308147 | 0.63 | 0.02 |
| yellow | 2025 | 4 | 1094213 | 2.25 | 0.02 |
| yellow | 2025 | 5 | 3 | 0.0 | 0.0 |
| green | 2026 | 1 | 219974 | 65.25 | 3.84 |

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### sql/eda/04_distribuciones.sql
| tipo | anio_archivo | viajes | distancia_p50 | distancia_p95 | distancia_p99 | total_p50 | total_p95 | total_p99 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| green | 2024 | 660198 | 1.87 | 8.49 | 16.4 | 19.32 | 56.25 | 95.8 |
| yellow | 2024 | 41169664 | 1.76 | 14.3 | 20.05 | 21.12 | 82.69 | 104.88 |
| green | 2025 | 591354 | 1.98 | 9.79 | 17.6 | 20.04 | 58.5 | 99.48 |
| yellow | 2025 | 48722573 | 1.85 | 12.75 | 19.55 | 21.54 | 77.94 | 104.88 |
| green | 2026 | 337100 | 2.07 | 10.52 | 17.76 | 20.5 | 57.72 | 97.2 |
| yellow | 2026 | 29703338 | 1.86 | 12.3 | 19.5 | 23.69 | 77.75 | 105.75 |

### sql/eda/05_atipicos.sql
| tipo | anio_archivo | viajes | distancia_mayor_100 | total_mayor_500 | pasajeros_mayor_8 | propina_mayor_total | duracion_invalida |
| --- | --- | --- | --- | --- | --- | --- | --- |
| green | 2024 | 660218 | 235 | 36 | 48 | 4 | 2 |
| yellow | 2024 | 41169720 | 1613 | 831 | 36 | 80 | 1805 |
| green | 2025 | 591375 | 209 | 27 | 58 | 0 | 1346 |
| yellow | 2025 | 48722602 | 2870 | 1210 | 34 | 58 | 2587 |
| green | 2026 | 337114 | 72 | 21 | 31 | 1 | 9 |
| yellow | 2026 | 29703355 | 1223 | 791 | 6 | 140 | 273 |

### sql/eda/06_validacion_anios.sql
| anio_archivo | tipo | archivos | viajes | primera_fecha | ultima_fecha |
| --- | --- | --- | --- | --- | --- |
| 2024 | green | 12 | 660218 | 2008-12-31 00:00:00 | 2025-01-01 22:21:15 |
| 2024 | yellow | 12 | 41169720 | 2002-12-31 16:46:07 | 2026-06-26 23:53:12 |
| 2025 | green | 12 | 591375 | 2008-12-31 15:13:04 | 2026-01-01 21:09:39 |
| 2025 | yellow | 12 | 48722602 | 2007-12-05 18:45:00 | 2025-12-31 23:59:59 |
| 2026 | green | 8 | 337114 | 2008-12-31 17:35:31 | 2026-08-31 23:58:28 |
| 2026 | yellow | 8 | 29703355 | 2001-01-01 09:23:58 | 2026-08-31 23:59:59 |